In [1]:
from datasets import load_dataset
import pandas as pd




In [2]:
# Load CoS-E v1.11
CoSEdataset = load_dataset("cos_e", "v1.11")

In [3]:
type(CoSEdataset)

datasets.dataset_dict.DatasetDict

In [4]:
# Look at fisrt 5 examples
print(CoSEdataset['train'][:5])

{'id': ['6b819727eb8a670df26a7ffad036c119', '3a1b3c21a11f4ec53c166b0559df7369', '3c4ea48ce584895fa85d5ee204e2444e', '18ad84abec6d00c77d1f05ddbb027fee', 'c4494b402264250dc70931613d482295'], 'question': ['"There are 10 apples on an apple tree.  Three fall off.  Now there are X apples."  What is this an example of?', 'A John is a bum.  Much like the stereotype, he lives near this sort of transportation infrastructure. Where does he live?', 'A bad person places little value on being honest, acting without pretense or being what?', 'A bald eagle flies over St. Paul, where is it?', 'A battleship is a powerful vessel.  If you need something similar but faster, what would you use?'], 'choices': [['park', 'coloring book', 'garden center', 'math problem', 'gravity'], ['bus depot', 'beach', 'train station', 'bridge', 'bridge'], ['excellent', 'upright', 'premium', 'competent', 'sincere'], ['texas', 'thermal', 'minnesota', 'canada', 'photograph'], ['yatch', 'corvette', 'aircraft carrier', 'destroye

In [5]:
print(CoSEdataset["train"].features)

{'id': Value(dtype='string', id=None), 'question': Value(dtype='string', id=None), 'choices': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'answer': Value(dtype='string', id=None), 'abstractive_explanation': Value(dtype='string', id=None), 'extractive_explanation': Value(dtype='string', id=None)}


In [6]:
#Lets convert to a samll df 

df = pd.DataFrame(CoSEdataset['train'][:10])
df[['question', 'answer', 'abstractive_explanation']]

,question,answer,abstractive_explanation
0,"""There are 10 apples on an apple tree. Three ...",math problem,webmath is designed to help you solve
1,"A John is a bum. Much like the stereotype, he...",bridge,bums are well known to take up residence under...
2,A bad person places little value on being hone...,sincere,this word is most relavant
3,"A bald eagle flies over St. Paul, where is it?",minnesota,st.paul is a county in minnesota
4,A battleship is a powerful vessel. If you nee...,corvette,"if you need speed, corvette is the answer."
5,"A beaver is taking logs from a Pacific beach, ...",washington,washington is the only place in the list that ...
6,A big fountain was the center piece of the ren...,city,a big fountain was the center piece of the ren...
7,"A boss may like an employee's ambition, so the...",in charge of project,being ambitious means they will work hard to b...
8,A boy is leaving line because he's tired of li...,punishment,punishment for disobedience
9,A bride and groom are taking care of proposals...,marriage,the term bride and groom is mostly associated ...


In [7]:
from transformers import pipeline

generator = pipeline("text2text-generation", model="google/flan-t5-large", device_map="auto")

prompt = "Question: Why do people wear sunscreen? Answer briefly."
print(generator(prompt, max_new_tokens=50)[0]['generated_text'])


Device set to use cpu


sunscreen protects skin from the sun 's rays


In [8]:
import transformers, tokenizers, huggingface_hub
print("transformers", transformers.__version__)
print("tokenizers", tokenizers.__version__)
print("hub", huggingface_hub.__version__)
from transformers import pipeline


transformers 4.47.1
tokenizers 0.21.0
hub 0.24.0


In [1]:
import sys, site, pyarrow as pa
print("Python:", sys.version)
print("Executable:", sys.executable)
print("Site-packages:", site.getusersitepackages(), site.getsitepackages() if hasattr(site, "getsitepackages") else "N/A")
print("pyarrow version:", pa.__version__)
print("Has PyExtensionType:", hasattr(pa, "PyExtensionType"))

Python: 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
Executable: C:\Users\azure\AppData\Local\Programs\Python\Python39\python.exe
Site-packages: C:\Users\azure\AppData\Roaming\Python\Python39\site-packages ['C:\\Users\\azure\\AppData\\Local\\Programs\\Python\\Python39', 'C:\\Users\\azure\\AppData\\Local\\Programs\\Python\\Python39\\lib\\site-packages']
pyarrow version: 19.0.1
Has PyExtensionType: True


In [2]:
question = "Why do people drink water?"

# ---------- FIX: use ':' between key and value (not '=') ----------
prompts = {
    "answer_only": (
        "Question: Why do people wear sunscreen?\n"
        "ANSWER = It prevents sunburn and long-term skin damage.\n\n"
        "Question: Why exercise regularly?\n"
        "ANSWER = It improves cardiovascular health and mental well-being.\n\n"
        "Question: {question}\n"
        "ANSWER ="
    ),

"answer_conf_expl_single_shot": (
 "Instruction: Replace placeholders and output EXACTLY one JSON object, nothing else.\n\n"
    "TEMPLATE:\n"
    '{"answer":"<ANS>","confidence":<0.00>,"explanation":"<EXPL>"}\n\n'
    "Question: {question}\n"
    "Now fill <ANS>, <0.00>, <EXPL> and output JSON only."
)

}

In [3]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM  # for Mistral Decoder only model 

# Choose model
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.1"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)  # <- changed
model.eval()

# --- generation helper (returns decoded text and full outputs for diagnostics) ---
def generate_text(prompt, max_new_tokens=80):
    with torch.no_grad():
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            num_beams=3,                  # deterministic+better quality
            no_repeat_ngram_size=3,
            return_dict_in_generate=True,
            output_scores=True,
            eos_token_id=getattr(tokenizer, "eos_token_id", None) or model.config.eos_token_id,
            pad_token_id=(getattr(tokenizer, "pad_token_id", None)
                          or getattr(tokenizer, "eos_token_id", None)
                          or model.config.eos_token_id),
        )
        decoded = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return decoded, outputs, inputs

# --- (your parsing/diagnostics functions unchanged) ---
CONF_PAT = re.compile(r"""
    (?:
      CONFIDENCE\s*=\s*
    )?
    (
      0(?:\.\d+)?                  # 0 or 0.xxx
      |1(?:\.0+)?                  # 1 or 1.0
      |\.\d+                       # .82
      |\d{1,3}%                    # 82%
    )
""", re.IGNORECASE | re.VERBOSE)

def parse_prompt_confidence(text):
    if not text:
        return None
    m = CONF_PAT.search(text)
    if not m:
        return None
    s = m.group(1)
    if s.endswith('%'):
        try:
            v = float(s[:-1]) / 100.0
            return max(0.0, min(1.0, round(v, 4)))
        except:
            return None
    try:
        v = float(s)
    except:
        return None
    if v > 1.0 and v <= 100.0:
        v = v / 100.0
    v = max(0.0, min(1.0, v))
    return round(v, 4)

NUM_RE = re.compile(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b")

def parse_confidence_simple(decoded):
    m = NUM_RE.search(decoded)
    if not m:
        return None
    try:
        val = float(m.group(0))
        if 0.01 <= val <= 0.99:
            return round(val, 2)
    except:
        pass
    return None

def extract_answer_from_decoded(decoded, question):
    marker = f"Question: {question}"
    tail = decoded.split(marker)[-1] if marker in decoded else decoded
    for ln in tail.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        if ln.upper().startswith("ANSWER"):
            parts = ln.split("=", 1)
            if len(parts) > 1:
                return parts[1].strip().strip('"').strip("'")
            else:
                continue
        if re.search(r"[A-Za-z]", ln):
            return ln.strip().strip('"').strip("'")
    return ""

def compute_model_confidence_from_scores(outputs):
    scores = outputs.scores
    if not scores:
        return None
    gen_len = len(scores)
    seq = outputs.sequences[0]
    generated_token_ids = seq[-gen_len:]
    token_probs = []
    for i, logits_step in enumerate(scores):
        logits = logits_step[0]
        probs = torch.softmax(logits, dim=-1)
        token_id = int(generated_token_ids[i])
        token_probs.append(float(probs[token_id]))
    return sum(token_probs) / len(token_probs) if token_probs else None

def softmax_probs(logits):
    return torch.softmax(logits, dim=-1)

def entropy_from_probs(probs):
    eps = 1e-12
    p = probs.clamp_min(eps)
    return float(-(p * p.log()).sum().item())

def topk_from_probs(probs, k=5):
    top_p, top_i = torch.topk(probs, k)
    items = []
    for p, idx in zip(top_p.tolist(), top_i.tolist()):
        tok_str = tokenizer.decode([idx], skip_special_tokens=False)
        items.append((tok_str, idx, float(p)))
    return items

def print_generation_diagnostics(prompt_name, prompt, outputs, decoded_text):
    print("\n" + "="*80)
    print(f"[{prompt_name}] PROMPT:\n{prompt}")
    print("-"*80)
    print(f"[{prompt_name}] DECODED OUTPUT:\n{decoded_text}")
    print("-"*80)
    seq = outputs.sequences[0]
    print(f"[{prompt_name}] sequences[0] (token IDs):\n{seq.tolist()}")
    print(f"[{prompt_name}] sequences[0] length: {len(seq)} tokens")
    scores = outputs.scores
    gen_len = len(scores)
    print(f"[{prompt_name}] Number of generated steps (len(scores)): {gen_len}")
    if gen_len == 0:
        print(f"[{prompt_name}] No scores returned (empty generation).")
        print("="*80)
        return
    generated_token_ids = seq[-gen_len:]
    print(f"[{prompt_name}] Generated token IDs (aligned to scores):\n{generated_token_ids.tolist()}")
    generated_token_strs = [tokenizer.decode([int(tid)], skip_special_tokens=False) for tid in generated_token_ids]
    print(f"[{prompt_name}] Generated tokens (per step, may include specials):\n{generated_token_strs}")
    eos_id = getattr(model.config, "eos_token_id", None)
    if eos_id is not None:
        ends_with_eos = (int(generated_token_ids[-1]) == eos_id)
        print(f"[{prompt_name}] Ends with EOS? {ends_with_eos} (eos_token_id={eos_id})")
    print("-"*80)
    print(f"[{prompt_name}] PER-STEP DETAILS:")
    step_token_probs = []
    step_entropies = []
    for i, logits_step in enumerate(scores):
        logits = logits_step[0]
        probs = softmax_probs(logits)
        chosen_id = int(generated_token_ids[i])
        chosen_prob = float(probs[chosen_id])
        ent = entropy_from_probs(probs)
        topk = topk_from_probs(probs, k=5)
        step_token_probs.append(chosen_prob)
        step_entropies.append(ent)
        chosen_tok = tokenizer.decode([chosen_id], skip_special_tokens=False)
        print(f"\nStep {i:02d}:")
        print(f"  Chosen token: {repr(chosen_tok)} (id={chosen_id})")
        print(f"  Chosen token probability: {chosen_prob:.6f}")
        print(f"  Entropy of distribution: {ent:.6f} nats")
        print("  Top-5 candidates:")
        for j, (tok_str, tid, p) in enumerate(topk, start=1):
            print(f"    {j}. {repr(tok_str):>12}  (id={tid:>6})  p={p:.6f}")
    mean_token_prob = sum(step_token_probs) / len(step_token_probs) if step_token_probs else None
    mean_entropy = sum(step_entropies) / len(step_entropies) if step_entropies else None
    print("-"*80)
    print(f"[{prompt_name}] Aggregate proxy metrics:")
    if mean_token_prob is not None:
        print(f"  Mean chosen-token probability: {mean_token_prob:.6f}")
    if mean_entropy is not None:
        print(f"  Mean per-step entropy: {mean_entropy:.6f} nats")
    print("="*80)


# ----------------- main flow -----------------
# make sure prompts and question are defined earlier in your script
for name, p in prompts.items():
    print(f"\n--- {name} ---")
    prompt = p.replace("{question}", question)
    text, outputs, inputs = generate_text(prompt, max_new_tokens=80)
    print("Raw model output:\n", text)
    print("\n---Raw model ouput ends ---\n")

    print_generation_diagnostics(name, prompt, outputs, text)

    answer = extract_answer_from_decoded(text, question)
    print(f"[{name}] Extracted ANSWER: {answer}")

    parsed_conf = parse_confidence_simple(text)
    if parsed_conf is not None:
        print(f"[{name}] Parsed prompt-elicited confidence: {parsed_conf}")

    model_conf = compute_model_confidence_from_scores(outputs)
    if model_conf is not None:
        print(f"[{name}] Model-token-prob proxy confidence (mean chosen-token prob): {round(model_conf, 6)}")


Device: cpu


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]


--- answer_only ---


From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


Raw model output:
 Question: Why do people wear sunscreen?
ANSWER = It prevents sunburn and long-term skin damage.

Question: Why exercise regularly?
ANSWER = It improves cardiovascular health and mental well-being.

Question: Why do people drink water?
ANSWER = To stay hydrated, regulate body temperature, and transport nutrients throughout the body.

---Raw model ouput ends ---


[answer_only] PROMPT:
Question: Why do people wear sunscreen?
ANSWER = It prevents sunburn and long-term skin damage.

Question: Why exercise regularly?
ANSWER = It improves cardiovascular health and mental well-being.

Question: Why do people drink water?
ANSWER =
--------------------------------------------------------------------------------
[answer_only] DECODED OUTPUT:
Question: Why do people wear sunscreen?
ANSWER = It prevents sunburn and long-term skin damage.

Question: Why exercise regularly?
ANSWER = It improves cardiovascular health and mental well-being.

Question: Why do people drink water?
ANSW

## Lets try a quantized model ##

In [12]:


import re
import torch


import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# ---- SELECT BACKEND ----
QUANTIZED_BACKEND = "gptq"  # "gptq" for GPTQ path

# ---- MODEL IDENTIFIERS ----
HF_BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.1"      # tokenizer source
GPTQ_REPO    = "TheBloke/Mistral-7B-Instruct-v0.1-GPTQ"   # quantized model repo

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# --- regex & helpers (unchanged) ---
CONF_PAT = re.compile(r"""
    (?:
      CONFIDENCE\s*=\s*
    )?
    (
      0(?:\.\d+)?      |1(?:\.0+)?   |\.\d+   |\d{1,3}%
    )
""", re.IGNORECASE | re.VERBOSE)

def parse_prompt_confidence(text):
    if not text:
        return None
    m = CONF_PAT.search(text)
    if not m:
        return None
    s = m.group(1)
    if s.endswith('%'):
        try:
            v = float(s[:-1]) / 100.0
            return max(0.0, min(1.0, round(v, 4)))
        except:
            return None
    try:
        v = float(s)
    except:
        return None
    if v > 1.0 and v <= 100.0:
        v = v / 100.0
    v = max(0.0, min(1.0, v))
    return round(v, 4)

NUM_RE = re.compile(r"\b(0(?:\.\d+)?|1(?:\.0+)?)\b")

def parse_confidence_simple(decoded):
    m = NUM_RE.search(decoded)
    if not m:
        return None
    try:
        val = float(m.group(0))
        if 0.01 <= val <= 0.99:
            return round(val, 2)
    except:
        pass
    return None

def extract_answer_from_decoded(decoded, question):
    marker = f"Question: {question}"
    tail = decoded.split(marker)[-1] if marker in decoded else decoded
    for ln in tail.splitlines():
        ln = ln.strip()
        if not ln:
            continue
        if ln.upper().startswith("ANSWER"):
            parts = ln.split("=", 1)
            if len(parts) > 1:
                return parts[1].strip().strip('"').strip("'")
            else:
                continue
        if re.search(r"[A-Za-z]", ln):
            return ln.strip().strip('"').strip("'")
    return ""

# ---- GPTQ loader ----
from transformers import AutoTokenizer
from auto_gptq import AutoGPTQForCausalLM

tokenizer = AutoTokenizer.from_pretrained(HF_BASE_MODEL, use_fast=True)

# Ensure PAD token exists; reuse EOS as PAD if missing (common for Mistral/LLaMA)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

eos_token_id = tokenizer.eos_token_id
pad_token_id = tokenizer.pad_token_id or eos_token_id

model = AutoGPTQForCausalLM.from_quantized(
    GPTQ_REPO,
    device_map="auto" if torch.cuda.is_available() else "cpu",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    use_safetensors=True,
    trust_remote_code=True,
)
model.eval()

# ---- generation helpers (unchanged interface) ----
def generate_text(prompt, max_new_tokens=80):
    with torch.no_grad():
        target_device = "cuda" if torch.cuda.is_available() else "cpu"
        inputs = tokenizer(prompt, return_tensors="pt")
        inputs = {k: v.to(target_device) for k, v in inputs.items()}
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,       # deterministic
            num_beams=1,           # faster than 3; start simple
            no_repeat_ngram_size=3,
            return_dict_in_generate=True,
            output_scores=True,
            eos_token_id=eos_token_id,
            pad_token_id=pad_token_id,
        )
        decoded = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    return decoded, outputs, inputs

def softmax_probs(logits):
    return torch.softmax(logits, dim=-1)

def entropy_from_probs(probs):
    eps = 1e-12
    p = probs.clamp_min(eps)
    return float(-(p * p.log()).sum().item())

def topk_from_probs(probs, k=5):
    top_p, top_i = torch.topk(probs, k)
    items = []
    for p, idx in zip(top_p.tolist(), top_i.tolist()):
        tok_str = tokenizer.decode([idx], skip_special_tokens=False)
        items.append((tok_str, idx, float(p)))
    return items

def print_generation_diagnostics(prompt_name, prompt, outputs, decoded_text):
    print("\n" + "="*80)
    print(f"[{prompt_name}] PROMPT:\n{prompt}")
    print("-"*80)
    print(f"[{prompt_name}] DECODED OUTPUT:\n{decoded_text}")
    print("-"*80)
    seq = outputs.sequences[0]
    print(f"[{prompt_name}] sequences[0] (token IDs):\n{seq.tolist()}")
    print(f"[{prompt_name}] sequences[0] length: {len(seq)} tokens")
    scores = outputs.scores
    gen_len = len(scores)
    print(f"[{prompt_name}] Number of generated steps (len(scores)): {gen_len}")
    if gen_len == 0:
        print(f"[{prompt_name}] No scores returned (empty generation).")
        print("="*80)
        return
    generated_token_ids = seq[-gen_len:]
    print(f"[{prompt_name}] Generated token IDs (aligned to scores):\n{generated_token_ids.tolist()}")
    generated_token_strs = [tokenizer.decode([int(tid)], skip_special_tokens=False) for tid in generated_token_ids]
    print(f"[{prompt_name}] Generated tokens (per step, may include specials):\n{generated_token_strs}")
    eos_id = eos_token_id
    if eos_id is not None:
        ends_with_eos = (int(generated_token_ids[-1]) == eos_id)
        print(f"[{prompt_name}] Ends with EOS? {ends_with_eos} (eos_token_id={eos_id})")
    print("-"*80)
    print(f"[{prompt_name}] PER-STEP DETAILS:")
    step_token_probs = []
    step_entropies = []
    for i, logits_step in enumerate(scores):
        logits = logits_step[0]
        probs = softmax_probs(logits)
        chosen_id = int(generated_token_ids[i])
        chosen_prob = float(probs[chosen_id])
        ent = entropy_from_probs(probs)
        topk = topk_from_probs(probs, k=5)
        step_token_probs.append(chosen_prob)
        step_entropies.append(ent)
        chosen_tok = tokenizer.decode([chosen_id], skip_special_tokens=False)
        print(f"\nStep {i:02d}:")
        print(f"  Chosen token: {repr(chosen_tok)} (id={chosen_id})")
        print(f"  Chosen token probability: {chosen_prob:.6f}")
        print(f"  Entropy of distribution: {ent:.6f} nats")
        print("  Top-5 candidates:")
        for j, (tok_str, tid, p) in enumerate(topk, start=1):
            print(f"    {j}. {repr(tok_str):>12}  (id={tid:>6})  p={p:.6f}")
    mean_token_prob = sum(step_token_probs) / len(step_token_probs) if step_token_probs else None
    mean_entropy = sum(step_entropies) / len(step_entropies) if step_entropies else None
    print("-"*80)
    print(f"[{prompt_name}] Aggregate proxy metrics:")
    if mean_token_prob is not None:
        print(f"  Mean chosen-token probability: {mean_token_prob:.6f}")
    if mean_entropy is not None:
        print(f"  Mean per-step entropy: {mean_entropy:.6f} nats")
    print("="*80)

def compute_model_confidence_from_scores(outputs):
    scores = outputs.scores
    if not scores:
        return None
    gen_len = len(scores)
    seq = outputs.sequences[0]
    generated_token_ids = seq[-gen_len:]
    token_probs = []
    for i, logits_step in enumerate(scores):
        logits = logits_step[0]
        probs = torch.softmax(logits, dim=-1)
        token_id = int(generated_token_ids[i])
        token_probs.append(float(probs[token_id]))
    return sum(token_probs) / len(token_probs) if token_probs else None



# Optional warm-up (reduces first-token latency)
#_ = generate_text("Hello", max_new_tokens=1)

for name, p in prompts.items():
    print(f"\n--- {name} ---")
    prompt = p.replace("{question}", question)
    text, outputs, inputs = generate_text(prompt, max_new_tokens=80)
    print("Raw model output:\n", text)

    print_generation_diagnostics(name, prompt, outputs, text)

    answer = extract_answer_from_decoded(text, question)
    print(f"[{name}] Extracted ANSWER: {answer}")

    parsed_conf = parse_confidence_simple(text)
    if parsed_conf is not None:
        print(f"[{name}] Parsed prompt-elicited confidence: {parsed_conf}")

    model_conf = compute_model_confidence_from_scores(outputs)
    if model_conf is not None:
        print(f"[{name}] Model-token-prob proxy confidence (mean chosen-token prob): {round(model_conf, 6)}")



Device: cpu
Error importing huggingface_hub.hf_file_system: cannot import name 'EntryNotFoundError' from 'huggingface_hub.errors' (C:\Users\azure\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\errors.py)


ImportError: cannot import name 'EntryNotFoundError' from 'huggingface_hub.errors' (C:\Users\azure\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\errors.py)

In [8]:

import sys, site, importlib, inspect
print("Kernel Python:", sys.executable)
print("sys.version:", sys.version)
import huggingface_hub
print("huggingface_hub:", huggingface_hub.__version__)
import huggingface_hub.errors as e
print("errors.py path:", inspect.getsourcefile(e))


Kernel Python: C:\Users\azure\AppData\Local\Programs\Python\Python39\python.exe
sys.version: 3.9.13 (tags/v3.9.13:6de2ca5, May 17 2022, 16:36:42) [MSC v.1929 64 bit (AMD64)]
huggingface_hub: 0.24.0
errors.py path: C:\Users\azure\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\errors.py


In [11]:

# Uninstall old hub from the kernel's environment
%pip uninstall -y huggingface_hub huggingface-hub
%pip cache purge

# Install the matching versions
%pip install --upgrade --force-reinstall huggingface_hub==0.36.0 transformers==4.57.3 tokenizers==0.22.1


Note: you may need to restart the kernel to use updated packages.


Files removed: 0 (0 bytes)
Note: you may need to restart the kernel to use updated packages.


   ---------------------------------------- 0.0/566.1 kB ? eta -:--:--
   ---------------------------------------- 566.1/566.1 kB 9.8 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ---------------------------------------  11.8/12.0 MB 720.9 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 49.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 21.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/15.9 MB ? eta -:--:--
   ---------- ----------------------------- 4.2/15.9 MB ? eta -:--:--
   ------------------------------- -------- 12.6/15.9 MB 49.3 MB/s eta 0:00:01
   ---------------------------------------  15.7/15.9 MB 33.0 MB/s eta 0:00:01
   ---------------------------------------- 15.9/15.9 MB 20.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
    Uninstalling u

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.4.1 requires fsspec[http]<=2024.12.0,>=2023.1.0, but you have fsspec 2025.10.0 which is incompatible.
lamini 3.2.17 requires numpy<2.0.0, but you have numpy 2.0.2 which is incompatible.
gensim 4.3.3 requires numpy<2.0,>=1.18.5, but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.13.1 requires keras<2.14,>=2.13.1, but you have keras 2.10.0 which is incompatible.
tensorflow-intel 2.13.1 requires numpy<=1.24.3,>=1.22, but you have numpy 2.0.2 which is incompatible.
tensorflow-intel 2.13.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 3.19.6 which is incompatible.
tensorflow-intel 